In [1]:
import json

T = CartanType(['A', 3])
W = WeylGroup(T,prefix="s")
[s1, s2, s3] = W.simple_reflections()
KLP = json.load( open( "kl_polys/a3.json" ) )

WCR = WeylCharacterRing(T, style="coroots", base_ring=QQ)
R = WeightRing(WCR)
# EXPI MODIFIED, THIS CODE ONLY WORKS IN TYPE A
f = [w.coerce_to_sl() for w in WCR.fundamental_weights()]

n = len(W.simple_reflections())
e = s1^2
w0 = W.long_element()
S.<q> = LaurentPolynomialRing(QQ)
KL = KazhdanLusztigPolynomial(W,q)
q = 11

def Iw(w):
    a = []
    for i in range(1, n+1):
        if w.has_right_descent(i):
            a.append(i)
    return a

def Iwc(w):
    a = []
    for i in range(1, n+1):
        if not w.has_right_descent(i):
            #a.append(W.simple_reflections()[i])
            a.append(i)
    return a

def expI(l):
    a = R.one()
    for i in l:
        a = a * R(f[i-1])
    return a

def Cpsub_on_wr_element(w, v):
    interv = W.bruhat_interval(e, w)
    s = 0
    for y in interv:
        a = v.weyl_group_action(y)
        s += int(KLP[str((y, w))]) * a
    return s

def Cpsup_on_wr_element(w, v):
    interv = W.bruhat_interval(e, w0*w)
    s = 0
    for y in interv:
        a = v.weyl_group_action(w0*y)
        s += int(KLP[str((y, w0*w))]) * a
    return s

def fsub(w):
    return Cpsub_on_wr_element(w, expI(Iwc(w)))

def qfsub(w):
    return Cpsub_on_wr_element(w, expI(Iwc(w)).scale(q))

def fsup(w):
    return Cpsup_on_wr_element(w, expI(Iw(w)))

invols = []
for w in W:
    if w*w == e:
        invols.append(w)

def Iweight(w):
    wgt = 0
    for a in Iw(w):
        wgt += f[a-1]
    return wgt

rho = Iweight(w0)

def weightpairing(l):
    a = WCR.dot_reduce(l)
    if a[0] == 0:
        return 0
    else:
        return a[0]*WCR(a[1])

def pairing(a, b):
    #takes two elements of R and gives you an element of WCR
    b = b.scale(-1)
    d = dict(R(a*b))
    r = 0
    for l in d:
        r += d[l]*weightpairing(l)
    return r

def woke_weightpairing(l):
    a = WCR.dot_reduce(l - rho)
    if a[0] == 0:
        return 0
    else:
        return a[0]*WCR(a[1])

def woke_pairing(a, b):
    #takes two elements of R and gives you an element of WCR
    d = dict(R(a*b))
    r = 0
    for l in d:
        r += d[l]*woke_weightpairing(l)
    return r

invols = []
for w in W:
    if w*w == e:
        invols.append(w)

d_sub = {}
d_sup = {}

for i in range(len(W)):
    print(i + 1, "out of", len(W), end='\r')
    d_sub[i] = fsub(W[i])
    d_sup[i] = fsup(W[i])

print("Setup done, f_w, f^w are defined.")

Setup done, f_w, f^w are defined.


In [2]:
svk = [R(-1, -1, -1), R(0, -1, -1), R(-1, 0, -1), R(-1, -1, 0), R(-1, 0, -1) + R(1, -1, -1), 
       R(-1, 0, 0) + R(-1, 1, -2) + R(0, -1, -1), R(-2, 1, -1) + R(-1, -1, 0) + R(0, 0, -1), 
       R(0, -1, 0), R(-1, -1, 1) + R(-1, 0, -1), R(-2, 1, -1) + R(-1, -1, 0) + R(0, -1, 1) + 2*R(0, 0, -1) + R(1, -2, 0) + R(2, -1, -1), 
       R(0, 0, -1), R(-1, 1, -1) + R(0, -1, 0), R(-1, -1, 1) + R(-1, 0, -1) + R(0, 0, 0) + R(1, -2, 1) + R(1, -1, -1), 
       R(-1, -1, 2) + 2*R(-1, 0, 0) + R(-1, 1, -2) + R(0, -2, 1) + R(0, -1, -1) + R(1, -1, 0), R(-1, 0, 0), 
       R(-1, 1, -1) + R(0, -1, 0) + R(1, 0, -1), R(-2, 1, 0) + R(-1, -1, 1) + R(-1, 0, -1) + R(-1, 2, -1) + 3*R(0, 0, 0) + R(0, 1, -2) + R(1, -2, 1) + R(1, -1, -1), 
       R(-1, 0, 0) + R(1, -1, 0), R(0, -1, 1) + R(0, 0, -1), R(-1, 0, 1) + R(-1, 1, -1) + R(0, -1, 0), R(-1, 0, 0) + R(0, 1, -1) + R(1, -1, 0), 
       R(-1, 0, 1) + R(-1, 1, -1) + R(0, -1, 0) + R(1, -1, 1) + R(1, 0, -1), R(-1, 1, 0) + R(0, -1, 1) + R(0, 0, -1), R(0, 0, 0)]
svk_dual = [R(1, 1, 1), R(0, 2, 1) + R(1, 0, 2) + R(1, 1, 0), R(0, 1, 2) + R(0, 2, 0) + R(1, 0, 1) + R(2, 0, 2) + R(2, 1, 0), R(0, 1, 1) + R(1, 2, 0) + R(2, 0, 1), R(0, 1, 2) + R(0, 2, 0) + R(1, 0, 1), R(1, 0, 2) + R(1, 1, 0), R(0, 1, 1) + R(2, 0, 1), R(-1, 2, 1) + R(0, 0, 2) + R(0, 1, 0) + R(0, 3, 0) + 3*R(1, 1, 1) + R(1, 2, -1) + R(2, -1, 2) + R(2, 0, 0), R(0, 2, 0) + R(1, 0, 1) + R(2, 1, 0), R(0, 1, 1), R(0, 0, 3) + 2*R(0, 1, 1) + R(0, 2, -1) + R(1, -1, 2) + R(1, 0, 0) + R(2, 0, 1), R(0, 0, 2) + R(0, 1, 0) + R(1, 1, 1) + R(2, -1, 2) + R(2, 0, 0), R(0, 2, 0) + R(1, 0, 1), R(1, 1, 0), R(-1, 2, 0) + R(0, 0, 1) + R(1, 0, 2) + 2*R(1, 1, 0) + R(2, -1, 1) + R(3, 0, 0), R(0, 0, 2) + R(0, 1, 0), R(1, 0, 1), R(-1, 2, 0) + R(0, 0, 1) + R(1, 1, 0), R(0, 1, 1) + R(0, 2, -1) + R(1, 0, 0), R(0, 1, 0) + R(2, 0, 0), R(0, 0, 1), R(0, 1, 0), R(1, 0, 0), R(0, 0, 0)]
element_list = [
 [3, 2, 3, 1, 2, 3],
 [2, 3, 1, 2, 3],
 [3, 2, 1, 2, 3],
 [3, 2, 3, 1, 2],
 [3, 1, 2, 3],
 [2, 1, 2, 3],
 [3, 2, 3, 1],
 [2, 3, 1, 2],
 [3, 2, 1, 2],
 [3, 2, 3],
 [1, 2, 3],
 [2, 3, 1],
 [3, 1, 2],
 [2, 1, 2],
 [3, 2, 1],
 [2, 3],
 [3, 1],
 [3, 2],
 [1, 2],
 [2, 1],
 [3],
 [2],
 [1],
 []
]

svk_dual_dict = {}
svk_dict = {}
S = list(W.simple_reflections())
for i in range(len(element_list)):
    a = s1*s1
    for j in range(len(element_list[i])):
        a = a * S[element_list[i][j]-1]
    svk_dict[a] = svk[i]
    svk_dual_dict[a] = (-1)^(a.length()) * svk_dual[i]
print(svk_dict)
print()
print(svk_dual_dict)

{s1*s2*s3*s1*s2*s1: a3(-1,-1,-1), s1*s2*s3*s1*s2: a3(0,-1,-1), s1*s2*s3*s2*s1: a3(-1,0,-1), s2*s3*s1*s2*s1: a3(-1,-1,0), s1*s2*s3*s2: a3(-1,0,-1) + a3(1,-1,-1), s1*s2*s3*s1: a3(0,-1,-1) + a3(-1,1,-2) + a3(-1,0,0), s2*s3*s2*s1: a3(-1,-1,0) + a3(-2,1,-1) + a3(0,0,-1), s2*s3*s1*s2: a3(0,-1,0), s3*s1*s2*s1: a3(-1,-1,1) + a3(-1,0,-1), s2*s3*s2: a3(-1,-1,0) + a3(-2,1,-1) + a3(1,-2,0) + 2*a3(0,0,-1) + a3(0,-1,1) + a3(2,-1,-1), s1*s2*s3: a3(0,0,-1), s2*s3*s1: a3(0,-1,0) + a3(-1,1,-1), s3*s1*s2: a3(0,0,0) + a3(-1,-1,1) + a3(-1,0,-1) + a3(1,-2,1) + a3(1,-1,-1), s1*s2*s1: a3(0,-1,-1) + a3(0,-2,1) + a3(-1,1,-2) + 2*a3(-1,0,0) + a3(-1,-1,2) + a3(1,-1,0), s3*s2*s1: a3(-1,0,0), s2*s3: a3(0,-1,0) + a3(-1,1,-1) + a3(1,0,-1), s3*s1: 3*a3(0,0,0) + a3(-2,1,0) + a3(-1,-1,1) + a3(-1,0,-1) + a3(1,-2,1) + a3(1,-1,-1) + a3(-1,2,-1) + a3(0,1,-2), s3*s2: a3(-1,0,0) + a3(1,-1,0), s1*s2: a3(0,0,-1) + a3(0,-1,1), s2*s1: a3(0,-1,0) + a3(-1,1,-1) + a3(-1,0,1), s3: a3(-1,0,0) + a3(1,-1,0) + a3(0,1,-1), s2: a3(0,-1,0) 

In [3]:
sup_dual = {s1*s1: -1/24*R(1,-1,1) + 1/12*R(0,1,0) + 1/24*R(1,1,1), s3*s1: 1/2*R(0,1,0), s2*s3*s1*s2: 3/4*R(0,0,0) + 1/4*R(2,-1,0) + 1/4*R(1,1,-1) + 1/2*R(1,0,1) + 1/4*R(1,-2,1) + 1/4*R(-1,2,-1) + 1/4*R(-1,1,1) + 1/4*R(0,-1,2), s1*s2*s3*s1*s2*s1: R(0,0,0), s1*s2: 1/4*R(1,0,1) + 1/4*R(-1,1,1), s1*s2*s3*s1: 1/2*R(-1,0,1) + 1/2*R(1,-1,1) + 1/2*R(0,1,0), s3*s2: 1/4*R(1,1,-1) + 1/4*R(1,0,1), s2*s3*s2*s1: 1/2*R(1,0,-1) + 1/2*R(1,-1,1) + 1/2*R(0,1,0), s2*s1: 1/6*R(1,-1,2) + 1/6*R(0,1,1), s2*s3: 1/6*R(2,-1,1) + 1/6*R(1,1,0), s1*s2*s3*s2: 1/2*R(-1,1,0) + 1/2*R(1,0,0), s3*s1*s2*s1: 1/2*R(0,1,-1) + 1/2*R(0,0,1), s1: -1/6*R(0,1,1), s3: -1/6*R(1,1,0), s1*s2*s3*s1*s2: -1/2*R(0,-1,1) - 1/2*R(-1,1,0) - 1/2*R(1,0,0), s2*s3*s1*s2*s1: -1/2*R(1,-1,0) - 1/2*R(0,1,-1) - 1/2*R(0,0,1), s2: -1/4*R(1,0,1), s2*s3*s1: -1/2*R(1,-1,1) - 1/2*R(0,1,0), s3*s1*s2: -1/4*R(0,0,0) - 1/4*R(1,1,-1) - 1/4*R(1,0,1) - 1/4*R(-1,2,-1) - 1/4*R(-1,1,1), s1*s2*s3*s2*s1: -1/2*R(-1,1,-1) - 1/2*R(-1,0,1) - 1/2*R(1,0,-1) - 1/2*R(1,-1,1) - R(0,1,0), s1*s2*s1: -1/2*R(0,0,1), s1*s2*s3: -1/6*R(-2,1,1) - 1/3*R(0,0,1) - 1/6*R(-1,2,0) - 1/6*R(2,-1,1) - 1/6*R(1,1,0), s2*s3*s2: -1/2*R(1,0,0), s3*s2*s1: -1/6*R(1,1,-2) - 1/3*R(1,0,0) - 1/6*R(1,-1,2) - 1/6*R(0,2,-1) - 1/6*R(0,1,1)}

In [4]:
for w in W:
    print(w)
    print(svk_dict[w])
    print()

1
a3(0,0,0)

s3*s1
3*a3(0,0,0) + a3(-2,1,0) + a3(-1,-1,1) + a3(-1,0,-1) + a3(1,-2,1) + a3(1,-1,-1) + a3(-1,2,-1) + a3(0,1,-2)

s2*s3*s1*s2
a3(0,-1,0)

s1*s2*s3*s1*s2*s1
a3(-1,-1,-1)

s1*s2
a3(0,0,-1) + a3(0,-1,1)

s1*s2*s3*s1
a3(0,-1,-1) + a3(-1,1,-2) + a3(-1,0,0)

s3*s2
a3(-1,0,0) + a3(1,-1,0)

s2*s3*s2*s1
a3(-1,-1,0) + a3(-2,1,-1) + a3(0,0,-1)

s2*s1
a3(0,-1,0) + a3(-1,1,-1) + a3(-1,0,1)

s2*s3
a3(0,-1,0) + a3(-1,1,-1) + a3(1,0,-1)

s1*s2*s3*s2
a3(-1,0,-1) + a3(1,-1,-1)

s3*s1*s2*s1
a3(-1,-1,1) + a3(-1,0,-1)

s1
a3(0,0,-1) + a3(0,-1,1) + a3(-1,1,0)

s3
a3(-1,0,0) + a3(1,-1,0) + a3(0,1,-1)

s1*s2*s3*s1*s2
a3(0,-1,-1)

s2*s3*s1*s2*s1
a3(-1,-1,0)

s2
a3(0,-1,0) + a3(-1,1,-1) + a3(-1,0,1) + a3(1,0,-1) + a3(1,-1,1)

s2*s3*s1
a3(0,-1,0) + a3(-1,1,-1)

s3*s1*s2
a3(0,0,0) + a3(-1,-1,1) + a3(-1,0,-1) + a3(1,-2,1) + a3(1,-1,-1)

s1*s2*s3*s2*s1
a3(-1,0,-1)

s1*s2*s1
a3(0,-1,-1) + a3(0,-2,1) + a3(-1,1,-2) + 2*a3(-1,0,0) + a3(-1,-1,2) + a3(1,-1,0)

s1*s2*s3
a3(0,0,-1)

s2*s3*s2
a3(-1,-1,0) + a3(-

In [18]:
print(svk_dict[s1*s1])
print(svk_dual_dict[w0])
print(pairing(R(0,0,0), R(1,1,1)))

for w in W:
    for y in W:
        if pairing(svk_dict[w], svk_dual_dict[y]) != 0:
            print(w == y)
            #print(pairing(svk_dict[w], svk_dual_dict[y]))
            #print()

#ok WOW so they ARE dual. Hm. 
print(pairing(WCR(3,2,1)*R(5,4,2), R(1,6,0)) - pairing(R(5,4,2), R(1,6,0)) * WCR(3,2,1))
print(pairing(R(5,4,2), WCR(3,2,1)*R(1,6,0)) - pairing(R(5,4,2), R(1,6,0)) * WCR(3,2,1))

a3(0,0,0)
a3(1,1,1)
0
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
0
-A3(0,2,0) - A3(0,0,4) - A3(0,1,2) - 2*A3(1,2,1) - 2*A3(1,0,5) - 2*A3(1,1,3) - A3(0,3,2) - A3(0,1,6) - 2*A3(0,2,4) - A3(2,1,0) + A3(2,3,0) - A3(2,0,6) - 2*A3(2,1,4) + 2*A3(1,4,1) - A3(1,3,3) - A3(1,2,5) + A3(0,6,0) + A3(0,5,2) + A3(3,1,1) + A3(3,0,3) + A3(5,0,1) + 2*A3(3,3,1) - 2*A3(3,2,3) - A3(3,1,5) + A3(2,5,0) - A3(2,3,4) + 2*A3(4,2,0) + 2*A3(4,1,2) + A3(6,1,0) + A3(6,0,2) + A3(4,4,0) - A3(4,2,4) + 3*A3(5,2,1) + A3(7,1,1) + A3(6,3,0) + A3(6,2,2)


In [8]:
#Defining [q]^*f_w
q = 11

svk_q_dict = {}
for i in range(len(W)):
    svk_q_dict[W[i]] = svk_dict[W[i]].scale(q)

svk_q_dual_dict = {}
for i in range(len(W)):
    svk_q_dual_dict[W[i]] = svk_dual_dict[W[i]].scale(q)

print(svk_q_dict)
print(svk_q_dual_dict)

{1: a3(0,0,0), s3*s1: 3*a3(0,0,0) + a3(-22,11,0) + a3(-11,-11,11) + a3(-11,0,-11) + a3(11,-22,11) + a3(11,-11,-11) + a3(-11,22,-11) + a3(0,11,-22), s2*s3*s1*s2: a3(0,-11,0), s1*s2*s3*s1*s2*s1: a3(-11,-11,-11), s1*s2: a3(0,0,-11) + a3(0,-11,11), s1*s2*s3*s1: a3(0,-11,-11) + a3(-11,11,-22) + a3(-11,0,0), s3*s2: a3(-11,0,0) + a3(11,-11,0), s2*s3*s2*s1: a3(-11,-11,0) + a3(-22,11,-11) + a3(0,0,-11), s2*s1: a3(0,-11,0) + a3(-11,11,-11) + a3(-11,0,11), s2*s3: a3(0,-11,0) + a3(-11,11,-11) + a3(11,0,-11), s1*s2*s3*s2: a3(-11,0,-11) + a3(11,-11,-11), s3*s1*s2*s1: a3(-11,-11,11) + a3(-11,0,-11), s1: a3(0,0,-11) + a3(0,-11,11) + a3(-11,11,0), s3: a3(-11,0,0) + a3(11,-11,0) + a3(0,11,-11), s1*s2*s3*s1*s2: a3(0,-11,-11), s2*s3*s1*s2*s1: a3(-11,-11,0), s2: a3(0,-11,0) + a3(-11,11,-11) + a3(-11,0,11) + a3(11,0,-11) + a3(11,-11,11), s2*s3*s1: a3(0,-11,0) + a3(-11,11,-11), s3*s1*s2: a3(0,0,0) + a3(-11,-11,11) + a3(-11,0,-11) + a3(11,-22,11) + a3(11,-11,-11), s1*s2*s3*s2*s1: a3(-11,0,-11), s1*s2*s1: a3(0

In [16]:
def nqpair(w, y):
    #new/normalized q_pairing
    return pairing(svk_q_dict[w], svk_dual_dict[y])

table = []

Mw = {}

rev_dict = {}

sumtot = 0
for w in W:
    #if w.length() < (w*s1).length():
    if True:
        if nqpair(w,w) not in rev_dict:
            rev_dict[nqpair(w,w)] = [w]
        else:
            rev_dict[nqpair(w,w)].append(w)
        table.append([str(w), str(nqpair(w, w))])
        sumtot += nqpair(w,w)

for row in table:
    print('| {:20} | {:1}'.format(*row))

print(sumtot)
print("-------")
for val in rev_dict:
    print(val)
    print(rev_dict[val])

| 1                    | A3(0,0,0)
| s3*s1                | A3(10,0,10)
| s2*s3*s1*s2          | A3(0,10,0)
| s1*s2*s3*s1*s2*s1    | A3(10,10,10)
| s1*s2                | A3(10,0,0)
| s1*s2*s3*s1          | A3(10,10,0)
| s3*s2                | A3(0,0,10)
| s2*s3*s2*s1          | A3(0,10,10)
| s2*s1                | -A3(0,8,0) + A3(0,10,0)
| s2*s3                | -A3(0,8,0) + A3(0,10,0)
| s1*s2*s3*s2          | A3(9,0,9) + A3(10,0,10)
| s3*s1*s2*s1          | A3(9,0,9) + A3(10,0,10)
| s1                   | A3(10,0,0)
| s3                   | A3(0,0,10)
| s1*s2*s3*s1*s2       | A3(10,10,0)
| s2*s3*s1*s2*s1       | A3(0,10,10)
| s2                   | A3(0,10,0)
| s2*s3*s1             | A3(0,8,0) + A3(0,10,0)
| s3*s1*s2             | -A3(9,0,9) + A3(10,0,10)
| s1*s2*s3*s2*s1       | A3(10,0,10)
| s1*s2*s1             | A3(10,10,0)
| s1*s2*s3             | A3(10,0,0)
| s2*s3*s2             | A3(0,10,10)
| s3*s2*s1             | A3(0,0,10)
A3(0,0,0) + 3*A3(0,0,10) - A3(0,8,0) + 5*A3(0,10,

In [15]:
for w in W:
    print(w)
    print(fsup(w0*w*w0))
    print(svk_dict[w])
    print()

print("---")

for w in W:
    print(w)
    print(fsub(w0*w*w0))
    print(svk_dual_dict[w])
    print()

1
24*a3(0,0,0)
a3(0,0,0)

s3*s1
2*a3(-2,1,0) + 2*a3(-1,-1,1) + 4*a3(-1,0,-1) + 2*a3(1,-2,1) + 2*a3(1,-1,-1) + 2*a3(-1,2,-1) + 2*a3(0,1,-2)
3*a3(0,0,0) + a3(-2,1,0) + a3(-1,-1,1) + a3(-1,0,-1) + a3(1,-2,1) + a3(1,-1,-1) + a3(-1,2,-1) + a3(0,1,-2)

s2*s3*s1*s2
4*a3(0,-1,0)
a3(0,-1,0)

s1*s2*s3*s1*s2*s1
a3(-1,-1,-1)
a3(-1,-1,-1)

s1*s2
4*a3(0,-1,0) + 4*a3(-1,1,-1) + 4*a3(1,0,-1)
a3(0,0,-1) + a3(0,-1,1)

s1*s2*s3*s1
2*a3(-1,0,-1) + 2*a3(1,-1,-1)
a3(0,-1,-1) + a3(-1,1,-2) + a3(-1,0,0)

s3*s2
4*a3(0,-1,0) + 4*a3(-1,1,-1) + 4*a3(-1,0,1)
a3(-1,0,0) + a3(1,-1,0)

s2*s3*s2*s1
2*a3(-1,-1,1) + 2*a3(-1,0,-1)
a3(-1,-1,0) + a3(-2,1,-1) + a3(0,0,-1)

s2*s1
6*a3(-1,0,0) + 6*a3(1,-1,0)
a3(0,-1,0) + a3(-1,1,-1) + a3(-1,0,1)

s2*s3
6*a3(0,0,-1) + 6*a3(0,-1,1)
a3(0,-1,0) + a3(-1,1,-1) + a3(1,0,-1)

s1*s2*s3*s2
2*a3(0,-1,-1) + 2*a3(-1,1,-2)
a3(-1,0,-1) + a3(1,-1,-1)

s3*s1*s2*s1
2*a3(-1,-1,0) + 2*a3(-2,1,-1)
a3(-1,-1,1) + a3(-1,0,-1)

s1
6*a3(-1,0,0) + 6*a3(1,-1,0) + 6*a3(0,1,-1)
a3(0,0,-1) + a3(0,-1,1) + a

In [25]:
for w in W:
    print("---")
    print(w)
    print()
    for y in W:
        if woke_pairing(sup_dual[y], svk_dict[w]) != 0:
            print(y, woke_pairing(sup_dual[y], svk_dict[w]))

---
1

1 1/24*A3(0,0,0)
---
s3*s1

1 1/8*A3(0,0,0)
s3*s1 1/2*A3(0,0,0)
s1*s2*s3*s2*s1 -1/2*A3(0,0,0)
---
s2*s3*s1*s2

s2*s3*s1*s2 1/4*A3(0,0,0)
---
s1*s2*s3*s1*s2*s1

s1*s2*s3*s1*s2*s1 A3(0,0,0)
---
s1*s2

s2*s1 1/6*A3(0,0,0)
---
s1*s2*s3*s1

s3*s1*s2*s1 1/2*A3(0,0,0)
s1*s2*s3 1/6*A3(0,0,0)
---
s3*s2

s2*s3 1/6*A3(0,0,0)
---
s2*s3*s2*s1

s1*s2*s3*s2 1/2*A3(0,0,0)
s3*s2*s1 1/6*A3(0,0,0)
---
s2*s1

s1*s2 1/4*A3(0,0,0)
---
s2*s3

s3*s2 1/4*A3(0,0,0)
---
s1*s2*s3*s2

s2*s3*s2*s1 1/2*A3(0,0,0)
---
s3*s1*s2*s1

s1*s2*s3*s1 1/2*A3(0,0,0)
---
s1

s1 1/6*A3(0,0,0)
---
s3

s3 1/6*A3(0,0,0)
---
s1*s2*s3*s1*s2

s2*s3*s1*s2*s1 1/2*A3(0,0,0)
---
s2*s3*s1*s2*s1

s1*s2*s3*s1*s2 1/2*A3(0,0,0)
---
s2

s2*s3*s1*s2 -1/4*A3(0,0,0)
s2 1/4*A3(0,0,0)
---
s2*s3*s1

s3*s1*s2 1/4*A3(0,0,0)
---
s3*s1*s2

1 1/24*A3(0,0,0)
s2*s3*s1 1/2*A3(0,0,0)
---
s1*s2*s3*s2*s1

s1*s2*s3*s2*s1 1/2*A3(0,0,0)
---
s1*s2*s1

s2*s3 1/6*A3(0,0,0)
s1*s2*s1 1/2*A3(0,0,0)
s1*s2*s3 1/6*A3(0,0,0)
---
s1*s2*s3

s3*s2*s1 1/6*A3(0,0,0)
---
s2

In [ ]:
_